In [1]:
# ------------------------------------------------------------
# STEP 1: LOAD DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

url = "gs://agntworks-data-dev/sandbox/experiments/clean_flight_data_with_regime_v3.csv"

df = pd.read_csv(url, parse_dates=["dep_datetime"])

print("Shape:", df.shape)
df.head()

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


Shape: (10794, 12)


,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime
0,157565,42366991KODR3217,2016-01-01,2016-01-01 23:37:00,23:37:00,WASHINGTON_DC_CLUSTER,OTHER_CLUSTER,44251.20,11954.115640,2.850000,4194.426540,NORMAL
1,157565,420331372FOSJ3517,2016-01-02,2016-01-02 18:00:00,18:00:00,OTHER_CLUSTER,MIAMI_CLUSTER,44251.20,6291.639810,1.500000,4194.426540,PEAK_MIXED
2,157565,420331372FOSJ4217,2016-01-02,2016-01-02 20:42:00,20:42:00,MIAMI_CLUSTER,SAN_FRANCISCO_CLUSTER,44251.20,26005.444550,6.200000,4194.426540,PEAK_MIXED
3,162215,42125996CASG317,2016-01-10,2016-01-10 14:56:00,14:56:00,BOSTON_CLUSTER,OTHER_CLUSTER,76864.32,22011.546392,4.166667,5282.771134,HIGH_YIELD
4,162215,423761209HARN017,2016-01-10,2016-01-10 21:10:00,21:10:00,OTHER_CLUSTER,SAN_JUAN_CLUSTER,76864.32,6251.279175,1.183333,5282.771134,HIGH_YIELD


In [2]:
# Check for missing values across all columns
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100

# Combine into a summary table
data_quality_df = pd.DataFrame({
    'Null Count': null_counts,
    'Percentage (%)': null_percentages
}).sort_values(by='Null Count', ascending=False)

print("Data Quality Summary:")
print(data_quality_df[data_quality_df['Null Count'] > 0])

# Check for empty strings in the cluster columns specifically
print("\nEmpty string check for Clusters:")
print("Dep_cluster empties:", (df['Dep_cluster'] == '').sum())
print("Arr_cluster empties:", (df['Arr_cluster'] == '').sum())

Data Quality Summary:
Empty DataFrame
Columns: [Null Count, Percentage (%)]
Index: []

Empty string check for Clusters:
Dep_cluster empties: 0
Arr_cluster empties: 0


In [3]:
unique_clusters = df['Dep_cluster'].unique()
print(unique_clusters)

['WASHINGTON_DC_CLUSTER' 'OTHER_CLUSTER' 'MIAMI_CLUSTER' 'BOSTON_CLUSTER'
 'SAN_JUAN_CLUSTER' 'NEW_YORK_CLUSTER' 'SAN_FRANCISCO_CLUSTER'
 'VAIL_CLUSTER' 'ORLANDO_CLUSTER' 'HONOLULU_CLUSTER' 'NASSAU_CLUSTER'
 'CANCUN_CLUSTER' 'JACKSON_HOLE_CLUSTER' 'CHICAGO_CLUSTER'
 'HOUSTON_CLUSTER' 'DENVER_CLUSTER' 'ASPEN_CLUSTER' 'DALLAS_CLUSTER'
 'LAS_VEGAS_CLUSTER' 'LOS_ANGELES_CLUSTER' 'SAN_DIEGO_CLUSTER'
 'CABO_CLUSTER' 'BARCELONA_CLUSTER' 'MADRID_CLUSTER' 'PUNTA_CANA_CLUSTER'
 'PHOENIX_CLUSTER' 'SEATTLE_CLUSTER' 'ATLANTA_CLUSTER' 'MILAN_CLUSTER'
 'LONDON_CLUSTER' 'PARIS_CLUSTER' 'DUBAI_CLUSTER' 'KUWAIT_CLUSTER'
 'ROME_CLUSTER' 'SINGAPORE_CLUSTER' 'TOKYO_CLUSTER' 'FLORIDA_GULF_CLUSTER'
 'HUNTSVILLE_CLUSTER' 'DELHI_CLUSTER' 'BRANSON_CLUSTER']


In [6]:
# 1. Create the 'corridor' feature first as we discussed
df['corridor'] = df['Dep_cluster'] + " -> " + df['Arr_cluster']

# 2. Generate the cluster_stats (this fixes the NameError)
cluster_stats = df.groupby('Dep_cluster').agg(
    total_rev=('Revenue_allocated', 'sum'),
    flight_count=('Trip_Legs_ID', 'count')
).reset_index()

# 3. Calculate the 5% threshold
revenue_total = df['Revenue_allocated'].sum()
revenue_threshold_5pct = 0.05 * revenue_total

# 4. Filter for clusters with 80+ flights AND >= 5% revenue
significant_clusters_df = cluster_stats[
    (cluster_stats['flight_count'] >= 80) & 
    (cluster_stats['total_rev'] >= revenue_threshold_5pct)
].sort_values(by='total_rev', ascending=False)

print(f"Total Network Revenue: ${revenue_total:,.2f}")
print(f"5% Revenue Threshold: ${revenue_threshold_5pct:,.2f}")
print("\n--- Significant 'Power Clusters' (5% Level) ---")
print(significant_clusters_df)

# Store the list for our Timezone mapping
significant_clusters = significant_clusters_df['Dep_cluster'].tolist()

Total Network Revenue: $155,744,323.30
5% Revenue Threshold: $7,787,216.17

--- Significant 'Power Clusters' (5% Level) ---
              Dep_cluster     total_rev  flight_count
25       NEW_YORK_CLUSTER  2.574248e+07          1628
33  SAN_FRANCISCO_CLUSTER  1.723966e+07          1093
22          MIAMI_CLUSTER  1.441414e+07          1008
20    LOS_ANGELES_CLUSTER  1.121521e+07           827
3          BOSTON_CLUSTER  9.004575e+06           700
27          OTHER_CLUSTER  8.234762e+06           508


In [7]:
# 1. Create the raw 'corridor' feature
df['corridor'] = df['Dep_cluster'] + " -> " + df['Arr_cluster']

# 2. (Optional) Create a 'clean_corridor' for better labeling in charts
# This removes the word '_CLUSTER' so it looks like 'NEW_YORK -> SAN_FRANCISCO'
df['corridor_clean'] = df['corridor'].str.replace('_CLUSTER', '', regex=True)

# 3. Preview the new feature
print("New 'corridor' feature added:")
print(df[['Dep_cluster', 'Arr_cluster', 'corridor_clean']].head())

New 'corridor' feature added:
             Dep_cluster            Arr_cluster          corridor_clean
0  WASHINGTON_DC_CLUSTER          OTHER_CLUSTER  WASHINGTON_DC -> OTHER
1          OTHER_CLUSTER          MIAMI_CLUSTER          OTHER -> MIAMI
2          MIAMI_CLUSTER  SAN_FRANCISCO_CLUSTER  MIAMI -> SAN_FRANCISCO
3         BOSTON_CLUSTER          OTHER_CLUSTER         BOSTON -> OTHER
4          OTHER_CLUSTER       SAN_JUAN_CLUSTER       OTHER -> SAN_JUAN
